# Traffic Flow Capstone

In the second course module you encountered different aspects of numerical problems and solution techniques that interact with each other. You have the interaction of spatial and time discretization operating together, and the implications for stability of the numerical solution. You saw the effect of nonlinearity in the convection equation, and the interplay between diffusion and convection in Burgers' equation. This is now a broader foundation to build your knowledge of numerical techniques.

In this capstone, you will put all this to work in a new problem setting: a model of traffic flow. Picture a stretch of one-lane road that starts with light traffic followed by a short patch of higher density. As the patch moves downstream, how does it change? This traffic model will allow you to estimate minimum and average vehicle speeds and investigate how much the computed results depend on the grid and time step.

## From density to traffic flux

Let $x$ measure distance downstream and $t$ measure time. The traffic density $\rho(x,t)$ is the number of cars per kilometer of road. We model the local vehicle speed $u$ as a function of density alone. At very low density, drivers travel near the free-flow speed $u_{\max}$. At the maximum density $\rho_{\max}$, traffic is bumper to bumper and comes to a stop. Assume that the speed decreases linearly between these limits.

$$
\label{eq-traffic-vehicle-speed}
u(\rho)=u_{\max}\left(1-\frac{\rho}{\rho_{\max}}\right).
$$

The traffic flux $F$ is the number of cars passing a point per hour. Multiplying density in cars per kilometer by vehicle speed in kilometers per hour gives

$$
\label{eq-traffic-flux}
F(\rho)=\rho u(\rho)
=u_{\max}\rho\left(1-\frac{\rho}{\rho_{\max}}\right).
$$

The flux is zero on an empty road because there are no cars to pass, and zero at bumper-to-bumper density because the cars are stationary.

```{figure} ./figures/velocity_and_flux.png
:label: fig-traffic-velocity-and-flux
:alt: Vehicle speed decreases linearly to zero, while the flux rises to a peak and falls back to zero.

Vehicle speed $u$ and traffic flux $F$ as functions of density $\rho$. The flux reaches its maximum at $\rho^\star=\rho_{\max}/2$.
```

## How density changes

There are no ramps or other sources of cars along this road. The number of cars on any stretch increases when more cars enter than leave. As in [the convection lesson](./06-1d-convection.ipynb#same-equation-different-algorithms), we express this balance with a conservation equation.

$$
\label{eq-traffic-conservation}
\frac{\partial \rho}{\partial t}+\frac{\partial F(\rho)}{\partial x}=0.
$$

Where the density is smooth, the chain rule gives $\rho_t+a(\rho)\rho_x=0$, with

$$
\label{eq-traffic-characteristic-speed}
a(\rho)=\frac{dF}{d\rho}
=u_{\max}\left(1-\frac{2\rho}{\rho_{\max}}\right).
$$

Both $u$ and $a$ have units of kilometers per hour. $u$ describes how fast the cars move, while $a$, the *characteristic speed*, describes how a small change in density travels through traffic. It's the speed at which information, such as a green light or a braking car, propagates on the road. For densities below $\rho_{\max}/2$, this change travels downstream because $a>0$. We will use the conservation form in [Equation %s](#eq-traffic-conservation) to compute the evolution of the patch, whose initial density jumps at its two edges.

## Set up the two scenarios

The table gives the model parameters, baseline discretization, and speeds to report for each scenario.

| Setting | Scenario A | Scenario B |
| --- | --- | --- |
| Road length | 11 km | 11 km |
| Maximum density $\rho_{\max}$ | 250 cars/km | 250 cars/km |
| Free-flow speed $u_{\max}$ | 80 km/h | 136 km/h |
| Background and left inflow density | 10 cars/km | 20 cars/km |
| Dense-patch density | 50 cars/km | 50 cars/km |
| Baseline grid and time step | 51 points, 0.001 h | 51 points, 0.001 h |
| Speeds to report | Minimum at 0 min, average at 3 min, minimum at 6 min | Minimum at 0 min, average and minimum at 3 min |

In both scenarios, the road starts at $x=0$. Set the initial density to the dense-patch value from 2.2 km up to, but excluding, 4.4 km. Use the background density everywhere else. The comparison between scenarios reflects changes in both the free-flow speed and the background density.

The baseline grid includes both endpoints, so $\Delta x=0.22\ \mathrm{km}$ and the patch occupies the Python slice `10:20`. Each time you double the number of intervals, double both slice indices to keep the patch in the same place and preserve its width.

### Boundary conditions

The initial densities in both scenarios lie below half the maximum density. We will work in this range, where information enters at the left end of the road. Hold $\rho(0,t)$ at the scenario's background density throughout the run. The right end is an outflow boundary. Its density evolves with the solution.

### Speeds to report

Use kilometers and hours in the calculation. You will evaluate the solution at the specified times (3 minutes is 0.05 h and 6 minutes is 0.10 h), and convert the vehicle speeds to meters per second for reporting.

For the minimum, use the smallest vehicle speed on the grid. For the average, take the arithmetic mean of the vehicle speeds at all grid points, including the endpoints. This is a spatial sample average. Each grid point carries the same weight regardless of the local number of cars.

## On paper — derive and check the traffic update

Before running the simulation, work through the discrete update of the density on paper. Use the four headings below to organize your notes. Keep your equations, units, and intermediate values so you can retrace your reasoning during a debrief of your work.

1. **Speeds and units.** Write down the units of $\rho$, $u$, $F$, and $a$. Differentiate $F(\rho)$ to recover $a(\rho)$. For each scenario, use the range from background density to dense-patch density to find the sign and largest value of $a$. Explain which neighbor supplies information to an updated point. Calculate the initial minimum vehicle speed in each scenario and show the conversion to meters per second.
2. **Discrete update.** Number the grid points from $0$ at the inlet to $N$ at the outlet. Starting from [Equation %s](#eq-traffic-conservation), use a forward difference in time and a backward difference of the flux. Solve for $\rho_i^{n+1}$. Mark the time level of every quantity on the right-hand side, identify which nodes you update, and write down the value you hold fixed. Use the largest characteristic speed to write the CFL restriction and check the baseline time step in both scenarios.
3. **Vehicle balance.** Multiply the change at each updated node by $\Delta x$ and add over nodes $1$ through $N$. Write out the first and last flux differences and mark the terms that cancel. Express the change in the discrete vehicle total in terms of the two remaining fluxes.
4. **One-step prediction.** Use a separate three-point example with Scenario A's speed and maximum density, $\Delta x=1\ \mathrm{km}$, $\Delta t=0.005\ \mathrm{h}$, and old densities $[10,50,20]$ cars/km from left to right. Hold the inlet at $10$ cars/km. List the three old fluxes, calculate the new density at each updated node, and use the inlet and outlet fluxes to predict whether the discrete vehicle total increases or decreases. Compare that prediction with the change calculated from your updated densities.

Keep your first attempt. When you reach the worked update and supplied code below, compare them with your notes and record any correction beside your original reasoning. Use the supplied update for the simulation.